## 依赖导入

数值方法一节：二维 Poisson 方程五点格式差分（4×4 小网格直接解线性方程组、50×50 大网格用 scipy.sparse 组装并 `np.linalg.solve` 求解）与一维反应-扩散方程的隐式格式时间推进；解以 3D 曲面展示。导入 matplotlib、numpy 与 scipy.sparse。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import sparse

## 全局绘图配置

设置全局 `plt.rc`：启用 TeX 渲染数学公式、字号 15，使后续曲面图的公式坐标标签正常显示。

In [ ]:
plt.rc("text", usetex=True)
plt.rc("font", size=15)

## 15.4  五点差分格式的直接线性求解（4×4 例）

定解函数 $f(x,y)=\ln((1+x)^2+y^2)$ 给出边界值：单位正方形步长 $h=1/3$，4×4 节点网格 `u = np.zeros((4, 4))`（m=n=4），u[0,:]、u[-1,:] 两行与 u[:,0]、u[:,-1] 两列分别在 x=0/1、y=0/1 边界上采样 f。内部 4 个节点满足五点方程 $-4u+\text{四邻和}=\text{边界贡献的相反数}$：按顺序 [u(1,0)+u(0,1), u(3,1)+u(2,0), u(1,3)+u(0,2), u(2,3)+u(3,2)] 取负得常数项 b，五点系数矩阵 a 主对角 -4、邻点 1，`np.linalg.inv(a) @ b` 解出内部节点值并 round 4 位打印（源程序 ex17_4.py）。

In [ ]:
def f(x, y):
    return np.log((1 + x) ** 2 + y**2)


u = np.zeros((4, 4))
m = 4
n = 4
h = 1 / 3
u[0, :] = f(0, np.arange(m) * h)
u[-1, :] = f(1, np.arange(m) * h)
u[:, 0] = f(np.arange(n) * h, 0)
u[:, -1] = f(np.arange(n) * h, 1)
b = -np.array(
    [u[1, 0] + u[0, 1], u[3, 1] + u[2, 0], u[1, 3] + u[0, 2], u[2, 3] + u[3, 2]]
)
a = np.array([[-4, 1, 1, 0], [1, -4, 0, 1], [1, 0, -4, 1], [0, 1, 1, -4]])
x = np.linalg.inv(a) @ b
print("解 x:", np.round(x, 4))

## 15.5  五点格式解 Poisson 方程的稀疏组装与曲面

$M=50$，把 $[0,1]^2$ 等分（hx=hy=1/(M-1)），`X, Y = np.meshgrid(x1, y1)` 生成网格数据。三个函数：`rhs_func` 取源项 $20\cos(3\pi x)\sin(2\pi y)$ 在内部节点的值并展开成 ((M-2)²,) 数组（注意其内部直接引用全局 X、Y 而非形参）；`bc_dirichlet` 构造四条 Dirichlet 边界（左 $y^2$、右 1、顶 $x^3$、底 1），上下边界写入 g1 的固定位置，左右边界经 `sparse.kron` 与选择向量作用后并入右端向量 g，返回 g 与四段边界值；`generate_lhs_matrix` 用 `sparse.diags` 组装五点格式系数矩阵：主对角 $2(1+\alpha)$（$\alpha=h_x^2/h_y^2$）、上下对角 -1，块间耦合 $-I$ 由 `sparse.kron` 实现。右端项 `rhs = frhs·hx² + g`，`np.linalg.solve(A, rhs)` 解出内部值，转置 reshape 填回 U 并补齐四边边界，打印数值解。最后 `projection="3d"` 的 `plot_surface`（coolwarm 色图）配合 `tight_layout`、`view_init(20, -106)` 展示曲面（源程序 ex17_5.py）。

In [ ]:
def rhs_func(x, y, M):
    g = (
        20 * np.cos(3 * np.pi * X[1:-1, 1:-1]) * np.sin(2 * np.pi * Y[1:-1, 1:-1])
    ).flatten()
    f = [g[i :: M - 2] for i in range(M - 2)]
    return np.array(f).flatten()


def bc_dirichlet(x, y, M):
    lBC = y[:, 0] ** 2
    leftBC = lBC[1 : M - 1]
    rBC = np.ones(M)
    rightBC = rBC[1 : M - 1]
    tBC = x[0, :] ** 3
    topBC = tBC[1 : M - 1]
    bBC = np.ones(M)
    bottomBC = bBC[1 : M - 1]
    g1 = np.zeros((M - 2) ** 2)
    for i in range(M - 2):
        g1[(M - 2) * i] = topBC[i]
    for j in range(M - 2):
        g1[(M - 2) * (j + 1) - 1] = bottomBC[j]
    k1 = np.zeros((len(leftBC), 1))
    k1[0] = 1.0
    leftBCk = sparse.kron(k1, leftBC).toarray().flatten()
    k2 = np.zeros((len(rightBC), 1))
    k2[-1] = 1.0
    rightBCk = sparse.kron(k2, rightBC).toarray().flatten()
    g = g1 + leftBCk + rightBCk
    return [g, lBC, tBC, rBC, bBC]


def generate_lhs_matrix(M, hx, hy):
    alpha = hx**2 / hy**2
    n = M - 2
    main_diag = 2 * (1 + alpha) * np.ones(n)
    off_diag = -1 * np.ones(n)
    diagonals = [main_diag, off_diag, off_diag]
    B = sparse.diags(diagonals, [0, -1, 1], shape=(n, n)).toarray()
    C = sparse.diags([-1 * np.ones(M)], [0], shape=(n, n)).toarray()
    e1 = np.eye(n)
    A1 = sparse.kron(e1, B).toarray()
    e2 = sparse.diags([np.ones(M), np.ones(M)], [-1, 1], shape=(n, n)).toarray()
    A2 = sparse.kron(e2, C).toarray()
    return A1 + A2


M = 50
(x0, xf) = (0.0, 1.0)
(y0, yf) = (0.0, 1.0)
hx = (xf - x0) / (M - 1)
hy = (yf - y0) / (M - 1)
x1 = np.linspace(x0, xf, M)
y1 = np.linspace(y0, yf, M)
X, Y = np.meshgrid(x1, y1)
frhs = rhs_func(X, Y, M)
fbc = bc_dirichlet(X, Y, M)
rhs = frhs * (hx**2) + fbc[0]
A = generate_lhs_matrix(M, hx, hy)
V = np.linalg.solve(A, rhs)
V = V.reshape((M - 2, M - 2)).T
U = np.zeros((M, M))
U[1 : M - 1, 1 : M - 1] = V
U[:, 0] = fbc[1]
U[0, :] = fbc[2]
U[:, M - 1] = fbc[3]
U[M - 1, :] = fbc[4]
print("所求的数值解为：\n", U)

fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.plot_surface(X, Y, U, cmap=plt.cm.coolwarm)
ax.set(xlabel="$x$", ylabel="$y$", zlabel="$u$")
fig.tight_layout()
ax.view_init(20, -106)
plt.show()

## 15.6  一维反应-扩散方程的隐式差分格式

空间 M=50、时间 N=60 个节点；dx、dt 为空间/时间步长，扩散系数 D=0.1、反应率 $\alpha=-3$，网格比 $r=dt\,D/dx^2$、$s=dt\,\alpha$。隐式格式的系数矩阵 A 为三对角（主对角 $1+2r-s$、上下对角 $-r$），由 `sparse.diags` 构造。初值 $U[:,0]=4x-4x^2$，边界 $U[0,:]=U[-1,:]=0$。对每个时间层 k=1…N-1：把上一时刻层内值 b2 与边界贡献 [r·U[0,k], r·U[-1,k]]（中间插入零向量补成三对角相邻项）相加得常数项 b，`np.linalg.solve(A, b)` 解出当前层内部值。最后 `X, T = np.meshgrid(tspan, xspan)`，`projection="3d"` 的 `plot_surface` 绘制解曲面（coolwarm、无描边线宽、x 刻度 0~0.2）并 tight_layout（源程序 ex17_6.py）。

In [ ]:
M = 50
N = 60
x0 = 0
xL = 1
dx = (xL - x0) / (M - 1)
t0 = 0
tF = 0.2
dt = (tF - t0) / (N - 1)
D = 0.1
alpha = -3
r = dt * D / dx**2
s = dt * alpha
xspan = np.linspace(x0, xL, M)
tspan = np.linspace(t0, tF, N)
main_diag = (1 + 2 * r - s) * np.ones(M - 2)
off_diag = -r * np.ones(M - 3)
n = M - 2
diagonals = [main_diag, off_diag, off_diag]
A = sparse.diags(diagonals, [0, -1, 1], shape=(n, n)).toarray()

U = np.zeros((M, N))
U[:, 0] = 4 * xspan - 4 * xspan**2
U[0, :] = 0.0
U[-1, :] = 0.0
for k in range(1, N):
    c = np.zeros(M - 4)
    b1 = np.array([r * U[0, k], r * U[-1, k]])
    b1 = np.insert(b1, 1, c)
    b2 = U[1 : M - 1, k - 1]
    b = b1 + b2
    U[1 : M - 1, k] = np.linalg.solve(A, b)

X, T = np.meshgrid(tspan, xspan)
fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.plot_surface(X, T, U, linewidth=0, cmap=plt.cm.coolwarm)
ax.set(xticks=[0, 0.05, 0.1, 0.15, 0.2], xlabel="$t$", ylabel="$x$", zlabel="$u$")
fig.tight_layout()
plt.show()